# GLINT: Multi-Seasonal Residual Gating in a Linear–Transformer for Cross-Dataset Multi-Horizon Water Demand Forecasting

This notebook is the canonical reproducibility implementation for **GLINT**. It reproduces the cross-dataset experiments on BWDF and SOWEKI, including the main baselines, ablations, statistical tests, efficiency measurements, and manuscript figures.

- Main trainable models: **Direct Linear, DLinear, LSTM, Patch Transformer, and GLINT**.
- GLINT hyperparameters: **frozen C1_small** selected using validation data only; no test-set tuning.
- Forecast horizons: **24, 72, 168, and 336 h** with a **336 h** historical lookback.
- Seeds: **42, 123, and 456**.
- The final claims in the manuscript follow the reported test evidence and controlled ablation results.


In [ ]:
# ============================================================
# 0. INSTALL + MOUNT GOOGLE DRIVE
# ============================================================
!pip -q install "wf4bwdf>=0.1.0" pyarrow openpyxl scipy scikit-learn tqdm

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# 1. IMPORTS + FROZEN FINAL CONFIGURATION
# ============================================================
from pathlib import Path
from datetime import datetime, timezone
import os, json, math, time, random, hashlib, warnings, re, copy
from urllib.parse import quote

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from scipy.stats import wilcoxon
import wf4bwdf as bwdf

warnings.filterwarnings("ignore")

PROJECT_DIR = Path("/content/drive/MyDrive/MFL_WaterDemand_FINAL_IJIES")
RAW_DIR = PROJECT_DIR / "data_raw"
PROC_DIR = PROJECT_DIR / "data_processed"
OUT_DIR = PROJECT_DIR / "final_outputs"
CKPT_DIR = PROJECT_DIR / "checkpoints"
LOG_DIR = PROJECT_DIR / "logs"
FIG_DIR = OUT_DIR / "figures"

for p in [PROJECT_DIR, RAW_DIR, PROC_DIR, OUT_DIR, CKPT_DIR, LOG_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123, 456]
HORIZONS = [24, 72, 168, 336]
MAX_H = 336
LOOKBACK = 336
WEEKLY_LAG = 168
DAILY_LAG = 24

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15

TRAIN_STRIDE = 6
VAL_STRIDE = 24
TEST_STRIDE = 24
MAX_TRAIN_WINDOWS_PER_SERIES = 1800
MAX_VAL_WINDOWS_PER_SERIES = 300
MAX_TEST_WINDOWS_PER_SERIES = 300

MIN_TARGET_OBS_FRACTION = 0.75
MIN_HORIZON_OBS_FRACTION = 0.50

# Manuscript forecast-example bank.
# Seed 42 is used only for visualization; quantitative conclusions use all seeds.
EXAMPLE_SEED = 42
EXAMPLE_WINDOWS_PER_SERIES = 2

BATCH_SIZE = 256
EPOCHS = 25
PATIENCE = 5
WEIGHT_DECAY = 1e-4
# Colab/Jupyter: multiprocessing workers can trigger
# '_MultiProcessingDataLoaderIter ... can only test a child process'.
# num_workers=0 is more stable and does not change the experiment definition.
NUM_WORKERS = 0
RESUME_CHECKPOINTS = True

# Frozen from the previous validation-only search.
FINAL_CFG = {
    "d_model": 32,
    "nhead": 4,
    "layers": 1,
    "dropout": 0.05,
    "context_dim": 96,
    "gate_dim": 48,
    "lr": 1e-3,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type != "cuda":
    print("WARNING: use a Colab GPU runtime for the FULL final run.")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

set_seed(SEEDS[0])

RUN_CONFIG = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "seeds": SEEDS,
    "horizons": HORIZONS,
    "lookback": LOOKBACK,
    "train_ratio": TRAIN_RATIO,
    "val_ratio": VAL_RATIO,
    "train_stride": TRAIN_STRIDE,
    "val_stride": VAL_STRIDE,
    "test_stride": TEST_STRIDE,
    "min_target_obs_fraction": MIN_TARGET_OBS_FRACTION,
    "final_model_config": FINAL_CFG,
}
with open(OUT_DIR / "FINAL_RUN_CONFIG.json", "w") as f:
    json.dump(RUN_CONFIG, f, indent=2)

print("Project:", PROJECT_DIR)

In [ ]:
# ============================================================
# 2. SELF-CONTAINED DATA DOWNLOAD + HOURLY STANDARDIZATION
# ============================================================

def retry_session():
    retry = Retry(
        total=8, connect=8, read=8, status=8,
        backoff_factor=2,
        status_forcelist=[408,429,500,502,503,504],
        allowed_methods=["GET"],
        raise_on_status=False,
        respect_retry_after_header=True,
    )
    s = requests.Session()
    a = HTTPAdapter(max_retries=retry, pool_connections=8, pool_maxsize=8)
    s.mount("https://", a)
    s.mount("http://", a)
    s.headers.update({"User-Agent":"MFL-WaterDemand-Final-IJIES/1.0"})
    return s

SESSION = retry_session()

def md5sum(path, block=1024*1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(block)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

# ---------- SOWEKI ----------
SOWEKI_DIR = RAW_DIR / "SOWEKI"
SOWEKI_DIR.mkdir(exist_ok=True)
ZENODO_RECORD = "19627517"
ZENODO_API = f"https://zenodo.org/api/records/{ZENODO_RECORD}"

def download_soweki():
    meta = SESSION.get(ZENODO_API, timeout=(30,120))
    if meta.status_code != 200:
        raise RuntimeError(f"Zenodo metadata failed: HTTP {meta.status_code}")
    rec = meta.json()
    files = [x for x in rec.get("files",[]) if x.get("key","").lower().endswith(".csv")]

    def numeric_key(item):
        nums = re.findall(r"\d+", item["key"])
        return int(nums[-1]) if nums else 999

    files = sorted(files, key=numeric_key)
    if len(files) != 11:
        print(f"WARNING: expected 11 SOWEKI CSV files, metadata reports {len(files)}.")

    paths = []
    for item in files:
        name = item["key"]
        dst = SOWEKI_DIR / name
        checksum = item.get("checksum","")
        expected = checksum.split(":",1)[1] if checksum.startswith("md5:") else None

        valid = dst.exists() and dst.stat().st_size > 0
        if valid and expected:
            valid = (md5sum(dst) == expected)

        if not valid:
            urls = [
                f"https://zenodo.org/records/{ZENODO_RECORD}/files/{quote(name)}?download=1",
                f"https://zenodo.org/api/records/{ZENODO_RECORD}/files/{quote(name)}/content",
                item.get("links",{}).get("content"),
            ]
            urls = [u for u in dict.fromkeys(urls) if u]
            ok = False
            last = None
            tmp = dst.with_suffix(dst.suffix + ".part")

            for url in urls:
                for attempt in range(3):
                    try:
                        with SESSION.get(url, stream=True, timeout=(30,300)) as r:
                            if r.status_code != 200:
                                raise requests.HTTPError(f"HTTP {r.status_code}")
                            with open(tmp, "wb") as f:
                                for chunk in r.iter_content(1024*1024):
                                    if chunk:
                                        f.write(chunk)
                        if tmp.stat().st_size == 0:
                            raise RuntimeError("Empty download")
                        if expected and md5sum(tmp) != expected:
                            raise RuntimeError("MD5 mismatch")
                        tmp.replace(dst)
                        ok = True
                        break
                    except Exception as e:
                        last = e
                        if tmp.exists():
                            tmp.unlink()
                        time.sleep(5 * (2**attempt))
                if ok:
                    break
            if not ok:
                raise RuntimeError(f"Unable to download {name}: {last}")
        paths.append(dst)

    with open(SOWEKI_DIR / "zenodo_metadata.json", "w") as f:
        json.dump(rec, f, indent=2)
    return paths

def infer_time_value(df):
    cols = list(df.columns)
    lower = {c:str(c).lower().strip() for c in cols}
    tc = [c for c in cols if any(k in lower[c] for k in ["timestamp","datetime","date_time","date","time"])]
    tcol = tc[0] if tc else cols[0]

    yc = [c for c in cols if c != tcol and any(k in lower[c] for k in ["demand","flow","consumption","water"])]
    if yc:
        ycol = yc[0]
    else:
        scores = {
            c: pd.to_numeric(df[c], errors="coerce").notna().mean()
            for c in cols if c != tcol
        }
        ycol = max(scores, key=scores.get)
    return tcol, ycol

def make_soweki_panel():
    paths = download_soweki()
    frames = []
    for fp in paths:
        df = pd.read_csv(fp)
        tcol, ycol = infer_time_value(df)
        tmp = pd.DataFrame({
            "ds": pd.to_datetime(df[tcol], errors="coerce", utc=True),
            "y": pd.to_numeric(df[ycol], errors="coerce"),
        }).dropna(subset=["ds"])
        # SOWEKI target is a rate (m^3/h) sampled each 15 min => hourly mean.
        tmp = (
            tmp.set_index("ds")[["y"]]
            .sort_index()
            .resample("1h").mean()
            .reset_index()
        )
        tmp["ds"] = tmp["ds"].dt.tz_convert("UTC").dt.tz_localize(None)
        nums = re.findall(r"\d+", fp.stem)
        sid = f"SOWEKI_{int(nums[-1]):02d}" if nums else fp.stem
        tmp["dataset"] = "SOWEKI"
        tmp["unique_id"] = sid
        frames.append(tmp[["dataset","unique_id","ds","y"]])
    return pd.concat(frames, ignore_index=True)

# ---------- BWDF ----------
BWDF_DIR = RAW_DIR / "BWDF"
BWDF_DIR.mkdir(exist_ok=True)
BWDF_PARQUET = BWDF_DIR / "bwdf_dma_inflows.parquet"

def acquire_bwdf():
    if not BWDF_PARQUET.exists():
        print("Loading BWDF via wf4bwdf...")
        obj = bwdf.load_complete_dataset()
        inflows = obj["dma-inflows"].copy()
        if not isinstance(inflows, pd.DataFrame):
            inflows = pd.DataFrame(inflows)
        inflows.to_parquet(BWDF_PARQUET)
    return pd.read_parquet(BWDF_PARQUET)

def make_bwdf_panel():
    df = acquire_bwdf()

    if isinstance(df.index, pd.DatetimeIndex):
        idx = pd.to_datetime(df.index, errors="coerce")
    else:
        tcols = [c for c in df.columns if any(k in str(c).lower() for k in ["time","date"])]
        if tcols:
            idx = pd.to_datetime(df[tcols[0]], errors="coerce")
            df = df.drop(columns=[tcols[0]])
        else:
            idx = pd.to_datetime(df.index, errors="coerce")

    try:
        if idx.tz is not None:
            idx = idx.tz_convert("UTC").tz_localize(None)
    except Exception:
        pass

    df.index = idx
    df = df[~df.index.isna()].sort_index()

    numeric = []
    for c in df.columns:
        s = pd.to_numeric(df[c], errors="coerce")
        # Retain DMA columns even when missingness exists.
        if s.notna().mean() >= 0.50:
            df[c] = s
            numeric.append(c)
    df = df[numeric]

    frames = []
    for i, c in enumerate(df.columns, start=1):
        tmp = pd.DataFrame({"ds":df.index, "y":df[c].values})
        tmp = (
            tmp.set_index("ds")[["y"]]
            .sort_index()
            .resample("1h").mean()
            .reset_index()
        )
        tmp["dataset"] = "BWDF"
        tmp["unique_id"] = f"BWDF_{i:02d}"
        frames.append(tmp[["dataset","unique_id","ds","y"]])

    out = pd.concat(frames, ignore_index=True)
    return out

BWDF_PROC = PROC_DIR / "bwdf_hourly_panel_final.parquet"
SOWEKI_PROC = PROC_DIR / "soweki_hourly_panel_final.parquet"

if BWDF_PROC.exists():
    bwdf_panel = pd.read_parquet(BWDF_PROC)
else:
    bwdf_panel = make_bwdf_panel()
    bwdf_panel.to_parquet(BWDF_PROC, index=False)

if SOWEKI_PROC.exists():
    soweki_panel = pd.read_parquet(SOWEKI_PROC)
else:
    soweki_panel = make_soweki_panel()
    soweki_panel.to_parquet(SOWEKI_PROC, index=False)

print("BWDF series:", bwdf_panel.unique_id.nunique(), "rows:", len(bwdf_panel))
print("SOWEKI series:", soweki_panel.unique_id.nunique(), "rows:", len(soweki_panel))

if bwdf_panel.unique_id.nunique() != 10:
    raise RuntimeError(
        f"FINAL protocol requires 10 BWDF series; found {bwdf_panel.unique_id.nunique()}. "
        "Inspect wf4bwdf columns before continuing."
    )
if soweki_panel.unique_id.nunique() != 11:
    raise RuntimeError(
        f"FINAL protocol requires 11 SOWEKI series; found {soweki_panel.unique_id.nunique()}."
    )

raw_panel = pd.concat([bwdf_panel, soweki_panel], ignore_index=True)
data_quality = (
    raw_panel.groupby(["dataset","unique_id"], as_index=False)
    .agg(
        start=("ds","min"),
        end=("ds","max"),
        n_hours=("y","size"),
        observed=("y",lambda s:s.notna().sum()),
        missing_pct=("y",lambda s:100*s.isna().mean()),
        mean=("y","mean"),
        std=("y","std"),
    )
)
data_quality.to_csv(OUT_DIR / "dataset_quality_raw.csv", index=False)
display(data_quality)

In [ ]:
# ============================================================
# 3. CAUSAL IMPUTATION + MASKED TARGET WINDOWS
# ============================================================

def causal_impute(raw, train_median):
    raw = np.asarray(raw, dtype=float)
    out = raw.copy()
    for i in range(len(out)):
        if np.isfinite(out[i]):
            continue
        if i >= WEEKLY_LAG and np.isfinite(out[i-WEEKLY_LAG]):
            out[i] = out[i-WEEKLY_LAG]
        elif i >= DAILY_LAG and np.isfinite(out[i-DAILY_LAG]):
            out[i] = out[i-DAILY_LAG]
        elif i > 0 and np.isfinite(out[i-1]):
            out[i] = out[i-1]
        else:
            out[i] = train_median
    return out

class PanelStore:
    def __init__(self, df):
        self.series_ids = sorted(df.unique_id.unique().tolist())
        self.raw = {}
        self.imputed_norm = {}
        self.observed_mask = {}
        self.target_norm = {}
        self.timestamps = {}
        self.scalers = {}
        self.splits = {}
        self.mase_denoms = {}

        for sid in self.series_ids:
            g = df[df.unique_id == sid].sort_values("ds").reset_index(drop=True)
            y = g.y.astype(float).to_numpy()
            ts = pd.to_datetime(g.ds).to_numpy()
            n = len(y)
            tr_end = int(n*TRAIN_RATIO)
            va_end = int(n*(TRAIN_RATIO+VAL_RATIO))

            train_obs = y[:tr_end][np.isfinite(y[:tr_end])]
            if len(train_obs) == 0:
                raise RuntimeError(f"No observed training target for {sid}")

            center = float(np.median(train_obs))
            q25, q75 = np.percentile(train_obs,[25,75])
            scale = float(q75-q25)
            if not np.isfinite(scale) or scale < 1e-8:
                scale = float(np.std(train_obs))
            if not np.isfinite(scale) or scale < 1e-8:
                scale = 1.0

            imp = causal_impute(y, center)
            obs = np.isfinite(y).astype(np.float32)

            imp_norm = ((imp-center)/scale).astype(np.float32)
            target_norm = ((y-center)/scale).astype(np.float32)

            # Weekly MASE denominator from OBSERVED training pairs only.
            tr = y[:tr_end]
            if len(tr) > WEEKLY_LAG:
                pair = np.isfinite(tr[WEEKLY_LAG:]) & np.isfinite(tr[:-WEEKLY_LAG])
                diffs = np.abs(tr[WEEKLY_LAG:][pair] - tr[:-WEEKLY_LAG][pair])
                denom = float(np.mean(diffs)) if len(diffs) else np.nan
            else:
                denom = np.nan
            if not np.isfinite(denom) or denom < 1e-8:
                denom = float(np.mean(np.abs(np.diff(train_obs)))) if len(train_obs)>1 else 1.0
            if not np.isfinite(denom) or denom < 1e-8:
                denom = 1.0

            self.raw[sid] = y
            self.imputed_norm[sid] = imp_norm
            self.observed_mask[sid] = obs
            self.target_norm[sid] = target_norm
            self.timestamps[sid] = ts
            self.scalers[sid] = (center, scale)
            self.splits[sid] = (tr_end, va_end, n)
            self.mase_denoms[sid] = denom

    def denorm(self, sid, arr):
        c, s = self.scalers[sid]
        return arr*s + c

def future_calendar(ts0, horizon=MAX_H):
    times = pd.date_range(pd.Timestamp(ts0), periods=horizon, freq="1h")
    h = times.hour.to_numpy(dtype=np.float32)
    d = times.dayofweek.to_numpy(dtype=np.float32)
    return np.stack([
        np.sin(2*np.pi*h/24),
        np.cos(2*np.pi*h/24),
        np.sin(2*np.pi*d/7),
        np.cos(2*np.pi*d/7),
    ], axis=1).astype(np.float32)

def valid_origins(store, sid, split, stride, cap):
    tr_end, va_end, n = store.splits[sid]
    if split == "train":
        start, stop = LOOKBACK, tr_end
    elif split == "val":
        start, stop = tr_end, va_end
    elif split == "test":
        start, stop = va_end, n
    else:
        raise ValueError(split)

    lo = max(LOOKBACK, start)
    hi = min(stop-MAX_H, n-MAX_H)
    if hi < lo:
        return []

    obs = store.observed_mask[sid]
    origins = []
    for t in range(lo, hi+1, stride):
        target_fraction = float(obs[t:t+MAX_H].mean())
        if target_fraction >= MIN_TARGET_OBS_FRACTION:
            origins.append(t)

    if cap is not None and len(origins) > cap:
        ii = np.linspace(0,len(origins)-1,cap).round().astype(int)
        origins = [origins[i] for i in ii]
    return origins

class WaterWindowDataset(Dataset):
    def __init__(self, store, split, stride, cap):
        self.store = store
        self.index = []
        for sidx, sid in enumerate(store.series_ids):
            origins = valid_origins(store, sid, split, stride, cap)
            self.index.extend((sidx,sid,t) for t in origins)
        if not self.index:
            raise RuntimeError(f"No windows for {split}")

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        sidx, sid, t = self.index[i]
        x = self.store.imputed_norm[sid][t-LOOKBACK:t].copy()
        xmask = self.store.observed_mask[sid][t-LOOKBACK:t].copy()

        y = self.store.target_norm[sid][t:t+MAX_H].copy()
        ymask = self.store.observed_mask[sid][t:t+MAX_H].copy()
        y = np.nan_to_num(y, nan=0.0).astype(np.float32)

        cal = future_calendar(self.store.timestamps[sid][t], MAX_H)

        return (
            torch.from_numpy(x),
            torch.from_numpy(xmask),
            torch.from_numpy(y),
            torch.from_numpy(ymask),
            torch.from_numpy(cal),
            torch.tensor(sidx,dtype=torch.long),
            int(t),
        )

def make_bundle(df):
    store = PanelStore(df)
    tr = WaterWindowDataset(store,"train",TRAIN_STRIDE,MAX_TRAIN_WINDOWS_PER_SERIES)
    va = WaterWindowDataset(store,"val",VAL_STRIDE,MAX_VAL_WINDOWS_PER_SERIES)
    te = WaterWindowDataset(store,"test",TEST_STRIDE,MAX_TEST_WINDOWS_PER_SERIES)

    loaders = {
        "train":DataLoader(tr,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=True),
        "val":DataLoader(va,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True),
        "test":DataLoader(te,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True),
    }
    return store,tr,va,te,loaders

bundles = {
    "BWDF":make_bundle(bwdf_panel),
    "SOWEKI":make_bundle(soweki_panel),
}

coverage_rows = []
for ds,bundle in bundles.items():
    store,tr,va,te,loaders = bundle
    for split,obj in [("train",tr),("val",va),("test",te)]:
        ids = [store.series_ids[x[0]] for x in obj.index]
        counts = pd.Series(ids).value_counts()
        for sid in store.series_ids:
            coverage_rows.append({
                "dataset":ds,"split":split,"unique_id":sid,
                "n_windows":int(counts.get(sid,0))
            })
    print(ds,"series",len(store.series_ids),"windows",len(tr),len(va),len(te))

coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(OUT_DIR/"window_coverage.csv",index=False)

# Hard guardrail: every final-test series must contribute at least one window.
for ds in ["BWDF","SOWEKI"]:
    te = coverage[(coverage.dataset==ds)&(coverage.split=="test")]
    missing = te[te.n_windows==0].unique_id.tolist()
    if missing:
        raise RuntimeError(f"{ds}: no final-test windows for {missing}")

In [ ]:
# ============================================================
# 4. MODELS
# ============================================================

class BaseForecastModel(nn.Module):
    def details(self, x, xmask, cal):
        return {"forecast":self.forward(x,xmask,cal)}

class DirectLinear(BaseForecastModel):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(LOOKBACK,MAX_H)
    def forward(self,x,xmask=None,cal=None):
        return self.linear(x)

class DLinear(BaseForecastModel):
    def __init__(self,kernel=25):
        super().__init__()
        self.kernel=kernel
        self.resid=nn.Linear(LOOKBACK,MAX_H)
        self.trend=nn.Linear(LOOKBACK,MAX_H)
    def forward(self,x,xmask=None,cal=None):
        pad=self.kernel//2
        tr=F.avg_pool1d(
            F.pad(x.unsqueeze(1),(pad,pad),mode="replicate"),
            kernel_size=self.kernel,stride=1
        ).squeeze(1)
        return self.resid(x-tr)+self.trend(tr)

class LSTMForecaster(BaseForecastModel):
    def __init__(self,hidden=64):
        super().__init__()
        self.lstm=nn.LSTM(input_size=2,hidden_size=hidden,batch_first=True)
        self.head=nn.Sequential(
            nn.LayerNorm(hidden),nn.Linear(hidden,hidden),nn.GELU(),
            nn.Dropout(FINAL_CFG["dropout"]),nn.Linear(hidden,MAX_H)
        )
    def forward(self,x,xmask,cal=None):
        inp=torch.stack([x,xmask],dim=-1)
        _,(h,_) = self.lstm(inp)
        return self.head(h[-1])

class MaskedPatchEncoder(nn.Module):
    def __init__(self,input_len,patch_len,stride,d_model,nhead,layers,dropout):
        super().__init__()
        self.patch_len=patch_len
        self.stride=stride
        self.n_patches=1+(input_len-patch_len)//stride
        self.proj=nn.Linear(2*patch_len,d_model)
        self.pos=nn.Parameter(torch.zeros(1,self.n_patches,d_model))
        nn.init.normal_(self.pos,std=.02)
        layer=nn.TransformerEncoderLayer(
            d_model=d_model,nhead=nhead,dim_feedforward=4*d_model,
            dropout=dropout,activation="gelu",batch_first=True,norm_first=True
        )
        self.enc=nn.TransformerEncoder(layer,num_layers=layers)
        self.norm=nn.LayerNorm(d_model)

    def forward(self,x,mask):
        px=x.unfold(1,self.patch_len,self.stride)
        pm=mask.unfold(1,self.patch_len,self.stride)
        z=self.proj(torch.cat([px,pm],dim=-1))
        z=z+self.pos[:,:z.size(1)]
        return self.norm(self.enc(z))

class PatchTransformer(BaseForecastModel):
    def __init__(self):
        super().__init__()
        c=FINAL_CFG
        self.enc=MaskedPatchEncoder(
            LOOKBACK,24,12,c["d_model"],c["nhead"],c["layers"],c["dropout"]
        )
        flat=self.enc.n_patches*c["d_model"]
        self.head=nn.Sequential(
            nn.Linear(flat,c["context_dim"]),nn.GELU(),nn.Dropout(c["dropout"]),
            nn.Linear(c["context_dim"],MAX_H)
        )
    def forward(self,x,xmask,cal=None):
        z=self.enc(x,xmask)
        return self.head(z.flatten(1))

class ResidualDLinearExpert(nn.Module):
    def __init__(self,kernel=25):
        super().__init__()
        self.kernel=kernel
        self.resid=nn.Linear(WEEKLY_LAG,MAX_H)
        self.trend=nn.Linear(WEEKLY_LAG,MAX_H)
    def forward(self,r):
        pad=self.kernel//2
        tr=F.avg_pool1d(
            F.pad(r.unsqueeze(1),(pad,pad),mode="replicate"),
            kernel_size=self.kernel,stride=1
        ).squeeze(1)
        return self.resid(r-tr)+self.trend(tr)

class ResidualTransformerExpert(nn.Module):
    def __init__(self,raw_mode=False):
        super().__init__()
        c=FINAL_CFG
        self.raw_mode=raw_mode
        input_len=LOOKBACK if raw_mode else WEEKLY_LAG
        self.local_len=72 if not raw_mode else 168

        self.local=MaskedPatchEncoder(
            self.local_len,12,6,c["d_model"],c["nhead"],c["layers"],c["dropout"]
        )
        self.long=MaskedPatchEncoder(
            input_len,24,12,c["d_model"],c["nhead"],c["layers"],c["dropout"]
        )
        flat=(self.local.n_patches+self.long.n_patches)*c["d_model"]
        self.context=nn.Sequential(
            nn.Linear(flat,c["context_dim"]),nn.GELU(),nn.Dropout(c["dropout"]),
            nn.LayerNorm(c["context_dim"])
        )
        self.head=nn.Sequential(
            nn.Linear(c["context_dim"],c["context_dim"]),nn.GELU(),
            nn.Dropout(c["dropout"]),nn.Linear(c["context_dim"],MAX_H)
        )

    def forward(self,seq,mask):
        zl=self.local(seq[:,-self.local_len:],mask[:,-self.local_len:])
        zg=self.long(seq,mask)
        ctx=self.context(torch.cat([zl.flatten(1),zg.flatten(1)],dim=1))
        return self.head(ctx),ctx

class GLINTModel(BaseForecastModel):
    def __init__(
        self,
        use_linear=True,
        use_transformer=True,
        static_gate=False,
        use_calendar=True,
        transformer_residual=True
    ):
        super().__init__()
        c=FINAL_CFG
        self.use_linear=use_linear
        self.use_transformer=use_transformer
        self.static_gate=static_gate
        self.use_calendar=use_calendar
        self.transformer_residual=transformer_residual

        self.linear_expert=ResidualDLinearExpert() if use_linear else None
        self.transformer_expert=(
            ResidualTransformerExpert(raw_mode=not transformer_residual)
            if use_transformer else None
        )

        self.expert_names=["weekly","daily"]
        if use_linear: self.expert_names.append("linear")
        if use_transformer: self.expert_names.append("transformer")
        n_exp=len(self.expert_names)

        if static_gate:
            init=torch.zeros(MAX_H,n_exp)
            init[:,0]=1.0
            self.static_logits=nn.Parameter(init)
        else:
            self.stats_proj=nn.Sequential(
                nn.Linear(7,c["gate_dim"]),nn.GELU(),nn.LayerNorm(c["gate_dim"])
            )
            if use_transformer:
                self.ctx_proj=nn.Sequential(
                    nn.Linear(c["context_dim"],c["gate_dim"]),nn.GELU(),
                    nn.LayerNorm(c["gate_dim"])
                )
            else:
                self.ctx_proj=None

            cal_dim=16 if use_calendar else 0
            if use_calendar:
                self.cal_proj=nn.Sequential(nn.Linear(4,16),nn.GELU())

            self.gate=nn.Sequential(
                nn.Linear(c["gate_dim"]+cal_dim,c["gate_dim"]),
                nn.GELU(),nn.Dropout(c["dropout"]),
                nn.Linear(c["gate_dim"],n_exp)
            )
            # Conservative initial preference for seasonal/linear structure.
            with torch.no_grad():
                last=self.gate[-1]
                last.weight.zero_()
                bias=[]
                for name in self.expert_names:
                    bias.append(
                        1.2 if name=="weekly" else
                        0.4 if name=="daily" else
                        0.9 if name=="linear" else
                        0.0
                    )
                last.bias.copy_(torch.tensor(bias,dtype=torch.float32))

    def details(self,x,xmask,cal):
        weekly=x[:,-WEEKLY_LAG:].repeat(1,2)[:,:MAX_H]
        daily=x[:,-DAILY_LAG:].repeat(1,math.ceil(MAX_H/DAILY_LAG))[:,:MAX_H]

        residual=x[:,-WEEKLY_LAG:]-x[:,-2*WEEKLY_LAG:-WEEKLY_LAG]
        residual_mask=xmask[:,-WEEKLY_LAG:]*xmask[:,-2*WEEKLY_LAG:-WEEKLY_LAG]

        experts=[weekly,daily]
        aux={}

        if self.use_linear:
            lin_resid=self.linear_expert(residual)
            e_lin=weekly+lin_resid
            experts.append(e_lin)
            aux["linear_expert"]=e_lin

        tr_ctx=None
        if self.use_transformer:
            if self.transformer_residual:
                tr_out,tr_ctx=self.transformer_expert(residual,residual_mask)
                e_tr=weekly+tr_out
            else:
                tr_out,tr_ctx=self.transformer_expert(x,xmask)
                e_tr=tr_out
            experts.append(e_tr)
            aux["transformer_expert"]=e_tr

        stack=torch.stack(experts,dim=-1)

        if self.static_gate:
            weights=torch.softmax(self.static_logits,dim=-1).unsqueeze(0).expand(x.size(0),-1,-1)
        else:
            stats=torch.stack([
                residual.mean(1),
                residual.std(1),
                residual[:,-24:].mean(1),
                residual[:,-24:].std(1),
                xmask.mean(1),
                residual_mask.mean(1),
                x[:,-24:].mean(1),
            ],dim=1)
            context=self.stats_proj(stats)
            if tr_ctx is not None:
                context=context+self.ctx_proj(tr_ctx)
            ctx=context.unsqueeze(1).expand(-1,MAX_H,-1)

            if self.use_calendar:
                calz=self.cal_proj(cal)
                gate_in=torch.cat([ctx,calz],dim=-1)
            else:
                gate_in=ctx
            weights=torch.softmax(self.gate(gate_in),dim=-1)

        pred=torch.sum(weights*stack,dim=-1)
        return {
            "forecast":pred,
            "weights":weights,
            "expert_names":self.expert_names,
            "expert_forecasts":aux,
        }

    def forward(self,x,xmask,cal):
        return self.details(x,xmask,cal)["forecast"]

def make_model(name):
    if name=="DirectLinear": return DirectLinear()
    if name=="DLinear": return DLinear()
    if name=="LSTM": return LSTMForecaster()
    if name=="PatchTransformer": return PatchTransformer()
    if name=="GLINT": return GLINTModel()
    if name=="Abl_NoTransformer": return GLINTModel(use_transformer=False)
    if name=="Abl_NoLinear": return GLINTModel(use_linear=False)
    if name=="Abl_StaticGate": return GLINTModel(static_gate=True)
    if name=="Abl_NoCalendar": return GLINTModel(use_calendar=False)
    if name=="Abl_RawTransformer": return GLINTModel(transformer_residual=False)
    raise KeyError(name)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for n in [
    "DirectLinear","DLinear","LSTM","PatchTransformer","GLINT",
    "Abl_NoTransformer","Abl_NoLinear","Abl_StaticGate",
    "Abl_NoCalendar","Abl_RawTransformer"
]:
    print(f"{n:22s} {count_params(make_model(n)):,} params")

In [ ]:
# ============================================================
# 5. MASKED LOSS + TRAIN / EVALUATE / EFFICIENCY
# ============================================================

def masked_huber(pred,target,mask):
    loss=F.smooth_l1_loss(pred,target,reduction="none")
    return (loss*mask).sum()/mask.sum().clamp_min(1.0)

def multi_horizon_loss(pred,target,mask):
    vals=[]
    for h in HORIZONS:
        vals.append(masked_huber(pred[:,:h],target[:,:h],mask[:,:h]))
    return torch.stack(vals).mean()

def model_loss(model,name,x,xmask,y,ymask,cal):
    if name.startswith("GLINT") or name.startswith("Abl_"):
        d=model.details(x,xmask,cal)
        main=multi_horizon_loss(d["forecast"],y,ymask)
        aux=0.0
        for _,ep in d["expert_forecasts"].items():
            aux=aux+0.05*multi_horizon_loss(ep,y,ymask)
        return main+aux
    pred=model(x,xmask,cal)
    return multi_horizon_loss(pred,y,ymask)

@torch.no_grad()
def validation_score(model,loader,store):
    model.eval()
    rows=[]
    for x,xmask,y,ymask,cal,sidx,origins in loader:
        x=x.to(DEVICE); xmask=xmask.to(DEVICE); cal=cal.to(DEVICE)
        pred=model(x,xmask,cal).cpu().numpy()
        y_np=y.numpy(); m_np=ymask.numpy(); s_np=sidx.numpy()

        for i in range(len(pred)):
            sid=store.series_ids[int(s_np[i])]
            c,sc=store.scalers[sid]
            yp=pred[i]*sc+c
            yt=y_np[i]*sc+c
            mask=m_np[i].astype(bool)
            for h in HORIZONS:
                mh=mask[:h]
                if mh.mean()<MIN_HORIZON_OBS_FRACTION:
                    continue
                mae=np.mean(np.abs(yp[:h][mh]-yt[:h][mh]))
                rows.append((sid,h,mae/store.mase_denoms[sid]))

    df=pd.DataFrame(rows,columns=["sid","h","MASE"])
    if df.empty:
        return np.inf
    macro=df.groupby(["sid","h"]).MASE.mean().mean()
    return float(macro)

def train_one(dataset_name,model_name,seed,bundle):
    store,tr,va,te,loaders=bundle
    ckpt=CKPT_DIR/f"{dataset_name}__{model_name}__seed{seed}.pt"
    hist_file=LOG_DIR/f"{dataset_name}__{model_name}__seed{seed}.csv"

    set_seed(seed)
    model=make_model(model_name).to(DEVICE)

    if RESUME_CHECKPOINTS and ckpt.exists():
        payload=torch.load(ckpt,map_location=DEVICE)
        model.load_state_dict(payload["model_state"])
        return model,payload.get("train_seconds",np.nan),payload.get("peak_train_gpu_mb",np.nan)

    opt=torch.optim.AdamW(model.parameters(),lr=FINAL_CFG["lr"],weight_decay=WEIGHT_DECAY)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="min",factor=.5,patience=2)

    best=np.inf; best_state=None; bad=0; history=[]
    if DEVICE.type=="cuda":
        torch.cuda.reset_peak_memory_stats()

    t0=time.perf_counter()

    for ep in range(1,EPOCHS+1):
        model.train()
        total=0.0; n=0
        bar=tqdm(loaders["train"],desc=f"{dataset_name} {model_name} s{seed} e{ep}",leave=False)

        for x,xmask,y,ymask,cal,sidx,origins in bar:
            x=x.to(DEVICE,non_blocking=True)
            xmask=xmask.to(DEVICE,non_blocking=True)
            y=y.to(DEVICE,non_blocking=True)
            ymask=ymask.to(DEVICE,non_blocking=True)
            cal=cal.to(DEVICE,non_blocking=True)

            opt.zero_grad(set_to_none=True)
            loss=model_loss(model,model_name,x,xmask,y,ymask,cal)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step()

            total+=loss.item()*x.size(0); n+=x.size(0)

        val=validation_score(model,loaders["val"],store)
        sch.step(val)

        history.append({
            "epoch":ep,
            "train_loss":total/max(n,1),
            "val_macro_MASE":val,
            "lr":opt.param_groups[0]["lr"],
        })
        print(
            f"{dataset_name} {model_name} seed={seed} ep={ep:02d} "
            f"train={history[-1]['train_loss']:.5f} val_MASE={val:.5f}"
        )

        if val<best-1e-4:
            best=val
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            bad=0
        else:
            bad+=1
            if bad>=PATIENCE:
                break

    train_seconds=time.perf_counter()-t0
    peak_mb=(torch.cuda.max_memory_allocated()/1024**2) if DEVICE.type=="cuda" else np.nan

    if best_state is not None:
        model.load_state_dict(best_state)

    pd.DataFrame(history).to_csv(hist_file,index=False)
    torch.save({
        "model_state":model.state_dict(),
        "best_val_macro_MASE":best,
        "train_seconds":train_seconds,
        "peak_train_gpu_mb":peak_mb,
        "seed":seed,
        "model_name":model_name,
        "dataset":dataset_name,
    },ckpt)

    return model,train_seconds,peak_mb

def metric_values(yt,yp,mask,denom):
    yt=yt[mask]; yp=yp[mask]
    mae=float(np.mean(np.abs(yp-yt)))
    rmse=float(np.sqrt(np.mean((yp-yt)**2)))
    smape=float(200*np.mean(np.abs(yp-yt)/(np.abs(yt)+np.abs(yp)+1e-8)))
    mase=float(mae/max(denom,1e-8))
    return mae,rmse,smape,mase

@torch.no_grad()
def evaluate_model(model,dataset_name,model_name,seed,bundle,save_examples=False):
    store,tr,va,te,loaders=bundle
    model.eval()

    if DEVICE.type=="cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    rows=[]; gate_rows=[]; example_rows=[]
    total_windows=0
    saved_example_counts={sid:0 for sid in store.series_ids}

    # warmup
    warm=next(iter(loaders["test"]))
    x,xmask,y,ymask,cal,sidx,origins=warm
    _=model(x.to(DEVICE),xmask.to(DEVICE),cal.to(DEVICE))
    if DEVICE.type=="cuda":
        torch.cuda.synchronize()

    t0=time.perf_counter()

    for batch_idx,(x,xmask,y,ymask,cal,sidx,origins) in enumerate(loaders["test"]):
        xd=x.to(DEVICE,non_blocking=True)
        md=xmask.to(DEVICE,non_blocking=True)
        cd=cal.to(DEVICE,non_blocking=True)

        if model_name=="GLINT":
            details=model.details(xd,md,cd)
            pred=details["forecast"].cpu().numpy()
            w=details["weights"].cpu().numpy()
            expert_names=details["expert_names"]
        else:
            pred=model(xd,md,cd).cpu().numpy()
            w=None; expert_names=None

        y_np=y.numpy()
        mask_np=ymask.numpy()
        x_np=x.numpy()
        s_np=sidx.numpy()
        o_np=origins.numpy()
        total_windows+=len(pred)

        # Weekly seasonal reference, available for every saved example.
        seasonal_norm=np.tile(x_np[:,-WEEKLY_LAG:],(1,2))[:,:MAX_H]

        for i in range(len(pred)):
            sid=store.series_ids[int(s_np[i])]
            c,sc=store.scalers[sid]
            yp=pred[i]*sc+c
            yt=y_np[i]*sc+c
            ys=seasonal_norm[i]*sc+c
            observed=mask_np[i].astype(bool)
            origin_time=pd.Timestamp(store.timestamps[sid][int(o_np[i])])

            for h in HORIZONS:
                mh=observed[:h]
                if mh.mean()<MIN_HORIZON_OBS_FRACTION:
                    continue
                mae,rmse,smape,mase=metric_values(
                    yt[:h],yp[:h],mh,store.mase_denoms[sid]
                )
                rows.append({
                    "dataset":dataset_name,"model":model_name,"seed":seed,
                    "unique_id":sid,"origin":origin_time,"horizon":h,
                    "MAE":mae,"RMSE":rmse,"sMAPE":smape,"MASE":mase,
                    "observed_fraction":float(mh.mean()),
                })

                if w is not None:
                    ww=w[i,:h].mean(axis=0)
                    gr={
                        "dataset":dataset_name,"seed":seed,
                        "unique_id":sid,"origin":origin_time,"horizon":h
                    }
                    for j,nm in enumerate(expert_names):
                        gr[f"w_{nm}"]=float(ww[j])
                    gate_rows.append(gr)

            # Save a compact bank for ALL horizons. This does not affect metrics.
            if save_examples and saved_example_counts[sid] < EXAMPLE_WINDOWS_PER_SERIES:
                for h in HORIZONS:
                    mh=observed[:h]
                    if mh.mean()<MIN_HORIZON_OBS_FRACTION:
                        continue

                    # Example-level metrics used later for representative selection.
                    ex_mae,ex_rmse,ex_smape,ex_mase=metric_values(
                        yt[:h],yp[:h],mh,store.mase_denoms[sid]
                    )

                    for k in range(h):
                        example_rows.append({
                            "dataset":dataset_name,
                            "model":model_name,
                            "seed":seed,
                            "unique_id":sid,
                            "origin":origin_time,
                            "horizon":h,
                            "step":k+1,
                            "actual":float(yt[k]) if observed[k] else np.nan,
                            "prediction":float(yp[k]),
                            "seasonal_naive":float(ys[k]),
                            "observed":int(observed[k]),
                            "example_MAE":ex_mae,
                            "example_RMSE":ex_rmse,
                            "example_sMAPE":ex_smape,
                            "example_MASE":ex_mase,
                        })

                saved_example_counts[sid]+=1

    if DEVICE.type=="cuda":
        torch.cuda.synchronize()
    infer_seconds=time.perf_counter()-t0
    peak_infer_mb=(torch.cuda.max_memory_allocated()/1024**2) if DEVICE.type=="cuda" else np.nan

    eff={
        "dataset":dataset_name,"model":model_name,"seed":seed,
        "parameters":count_params(model),
        "test_windows":total_windows,
        "inference_seconds":infer_seconds,
        "ms_per_window":1000*infer_seconds/max(total_windows,1),
        "windows_per_second":total_windows/max(infer_seconds,1e-9),
        "peak_inference_gpu_mb":peak_infer_mb,
    }
    return pd.DataFrame(rows),pd.DataFrame(gate_rows),pd.DataFrame(example_rows),eff

def seasonal_naive_eval(dataset_name,bundle):
    store,tr,va,te,loaders=bundle
    rows=[]
    t0=time.perf_counter()
    nwin=0

    for x,xmask,y,ymask,cal,sidx,origins in loaders["test"]:
        x_np=x.numpy(); y_np=y.numpy(); m_np=ymask.numpy()
        s_np=sidx.numpy(); o_np=origins.numpy()
        pred=np.tile(x_np[:,-WEEKLY_LAG:],(1,2))[:,:MAX_H]
        nwin+=len(pred)

        for i in range(len(pred)):
            sid=store.series_ids[int(s_np[i])]
            c,sc=store.scalers[sid]
            yp=pred[i]*sc+c
            yt=y_np[i]*sc+c
            observed=m_np[i].astype(bool)
            origin_time=pd.Timestamp(store.timestamps[sid][int(o_np[i])])

            for h in HORIZONS:
                mh=observed[:h]
                if mh.mean()<MIN_HORIZON_OBS_FRACTION:
                    continue
                mae,rmse,smape,mase=metric_values(
                    yt[:h],yp[:h],mh,store.mase_denoms[sid]
                )
                rows.append({
                    "dataset":dataset_name,"model":"SeasonalNaive168","seed":"det",
                    "unique_id":sid,"origin":origin_time,"horizon":h,
                    "MAE":mae,"RMSE":rmse,"sMAPE":smape,"MASE":mase,
                    "observed_fraction":float(mh.mean())
                })

    sec=time.perf_counter()-t0
    eff={
        "dataset":dataset_name,"model":"SeasonalNaive168","seed":"det",
        "parameters":0,"test_windows":nwin,"inference_seconds":sec,
        "ms_per_window":1000*sec/max(nwin,1),
        "windows_per_second":nwin/max(sec,1e-9),
        "peak_inference_gpu_mb":0.0,
        "train_seconds":0.0,
        "peak_train_gpu_mb":0.0,
    }
    return pd.DataFrame(rows),eff

In [ ]:
# ============================================================
# 6. FINAL MAIN MODEL RUN — 3 SEEDS
# ============================================================

MAIN_MODELS = [
    "DirectLinear",
    "DLinear",
    "LSTM",
    "PatchTransformer",
    "GLINT",
]

all_metrics=[]
all_gates=[]
all_examples=[]
efficiency_rows=[]

# deterministic baseline
for ds in ["BWDF","SOWEKI"]:
    met,eff=seasonal_naive_eval(ds,bundles[ds])
    all_metrics.append(met)
    efficiency_rows.append(eff)

for ds in ["BWDF","SOWEKI"]:
    for model_name in MAIN_MODELS:
        for seed in SEEDS:
            print("\n"+"="*90)
            print("MAIN",ds,model_name,"seed",seed)
            print("="*90)

            model,train_sec,peak_train=train_one(
                ds,model_name,seed,bundles[ds]
            )
            met,gates,examples,eff=evaluate_model(
                model,ds,model_name,seed,bundles[ds],
                save_examples=(model_name=="GLINT" and seed==EXAMPLE_SEED)
            )
            eff["train_seconds"]=train_sec
            eff["peak_train_gpu_mb"]=peak_train

            all_metrics.append(met)
            if len(gates): all_gates.append(gates)
            if len(examples): all_examples.append(examples)
            efficiency_rows.append(eff)

            model=model.cpu()
            if DEVICE.type=="cuda":
                torch.cuda.empty_cache()

main_window_metrics=pd.concat(all_metrics,ignore_index=True)

# IMPORTANT:
# SeasonalNaive168 uses a deterministic label while trainable models use
# numeric seeds. PyArrow cannot serialize a mixed int/string object column.
# Store ALL seed labels as strings consistently.
main_window_metrics["seed"] = main_window_metrics["seed"].astype(str)
main_window_metrics.to_parquet(
    OUT_DIR/"main_window_metrics.parquet",
    index=False
)

efficiency=pd.DataFrame(efficiency_rows)
if "seed" in efficiency.columns:
    efficiency["seed"] = efficiency["seed"].astype(str)
efficiency.to_csv(OUT_DIR/"efficiency_by_seed.csv",index=False)

gate_windows=pd.concat(all_gates,ignore_index=True) if all_gates else pd.DataFrame()
if len(gate_windows):
    gate_windows["seed"] = gate_windows["seed"].astype(str)
    gate_windows.to_csv(OUT_DIR/"msrhgt_gate_windows.csv",index=False)

examples=pd.concat(all_examples,ignore_index=True) if all_examples else pd.DataFrame()
if len(examples):
    examples["seed"] = examples["seed"].astype(str)
    examples.to_csv(OUT_DIR/"forecast_examples.csv",index=False)

print("Main metrics rows:",len(main_window_metrics))
display(efficiency.head())

In [ ]:
# ============================================================
# 7. FINAL ABLATION RUN — 3 SEEDS
# ============================================================

ABLATIONS = [
    "Abl_NoTransformer",
    "Abl_NoLinear",
    "Abl_StaticGate",
    "Abl_NoCalendar",
    "Abl_RawTransformer",
]

ablation_metrics=[]
ablation_eff=[]

for ds in ["BWDF","SOWEKI"]:
    for model_name in ABLATIONS:
        for seed in SEEDS:
            print("\n"+"="*90)
            print("ABLATION",ds,model_name,"seed",seed)
            print("="*90)

            model,train_sec,peak_train=train_one(
                ds,model_name,seed,bundles[ds]
            )
            met,gates,examples,eff=evaluate_model(
                model,ds,model_name,seed,bundles[ds]
            )
            eff["train_seconds"]=train_sec
            eff["peak_train_gpu_mb"]=peak_train

            ablation_metrics.append(met)
            ablation_eff.append(eff)

            model=model.cpu()
            if DEVICE.type=="cuda":
                torch.cuda.empty_cache()

abl_window_metrics=pd.concat(ablation_metrics,ignore_index=True)
abl_window_metrics["seed"] = abl_window_metrics["seed"].astype(str)
abl_window_metrics.to_parquet(
    OUT_DIR/"ablation_window_metrics.parquet",
    index=False
)

ablation_efficiency=pd.DataFrame(ablation_eff)
if "seed" in ablation_efficiency.columns:
    ablation_efficiency["seed"] = ablation_efficiency["seed"].astype(str)
ablation_efficiency.to_csv(OUT_DIR/"ablation_efficiency_by_seed.csv",index=False)

print("Ablation metrics rows:",len(abl_window_metrics))

In [ ]:
# ============================================================
# 8. AGGREGATE SEEDS FAIRLY + MAIN RESULTS + ABLATION
# ============================================================

def aggregate_results(window_df):
    # First rolling windows -> one score per series/seed/horizon.
    series_seed=(
        window_df
        .groupby(["dataset","model","seed","unique_id","horizon"],as_index=False)
        .agg(
            MAE=("MAE","mean"),
            RMSE=("RMSE","mean"),
            sMAPE=("sMAPE","mean"),
            MASE=("MASE","mean"),
            n_windows=("MAE","size"),
            observed_fraction=("observed_fraction","mean"),
        )
    )

    # Then series -> macro score per seed.
    seed_macro=(
        series_seed
        .groupby(["dataset","model","seed","horizon"],as_index=False)
        .agg(
            MAE=("MAE","mean"),
            RMSE=("RMSE","mean"),
            sMAPE=("sMAPE","mean"),
            MASE=("MASE","mean"),
            n_series=("unique_id","nunique"),
        )
    )

    # Finally seeds -> mean ± SD.
    final=(
        seed_macro
        .groupby(["dataset","model","horizon"],as_index=False)
        .agg(
            MAE=("MAE","mean"),
            MAE_seed_sd=("MAE","std"),
            RMSE=("RMSE","mean"),
            RMSE_seed_sd=("RMSE","std"),
            sMAPE=("sMAPE","mean"),
            sMAPE_seed_sd=("sMAPE","std"),
            MASE=("MASE","mean"),
            MASE_seed_sd=("MASE","std"),
            seeds=("seed","nunique"),
            n_series=("n_series","max"),
        )
    )
    # deterministic baseline has NaN seed SD
    for c in ["MAE_seed_sd","RMSE_seed_sd","sMAPE_seed_sd","MASE_seed_sd"]:
        final[c]=final[c].fillna(0.0)
    return series_seed,seed_macro,final

main_series_seed,main_seed_macro,main_results=aggregate_results(main_window_metrics)
abl_series_seed,abl_seed_macro,abl_results=aggregate_results(abl_window_metrics)

main_series_seed.to_csv(OUT_DIR/"main_series_seed_results.csv",index=False)
main_seed_macro.to_csv(OUT_DIR/"main_seed_macro_results.csv",index=False)
main_results.to_csv(OUT_DIR/"MAIN_RESULTS_FINAL.csv",index=False)

abl_series_seed.to_csv(OUT_DIR/"ablation_series_seed_results.csv",index=False)
abl_results.to_csv(OUT_DIR/"ABLATION_RESULTS_FINAL.csv",index=False)

main_results["rank_MASE"]=(
    main_results.groupby(["dataset","horizon"]).MASE
    .rank(method="min",ascending=True)
)

display(
    main_results.sort_values(["dataset","horizon","MASE"]).round(5)
)
print("\nAblation:")
display(
    pd.concat([
        main_results[main_results.model=="GLINT"],
        abl_results
    ],ignore_index=True)
    .sort_values(["dataset","horizon","MASE"])
    .round(5)
)

In [ ]:
# ============================================================
# 9. SERIES-LEVEL STATISTICS + EFFECT SIZE + BOOTSTRAP CI
# ============================================================

def holm_adjust(pvals):
    p=np.asarray(pvals,dtype=float)
    if len(p)==0: return np.array([])
    order=np.argsort(p)
    out=np.empty(len(p))
    running=0.0
    m=len(p)
    for rank,idx in enumerate(order):
        cand=(m-rank)*p[idx]
        running=max(running,cand)
        out[idx]=min(running,1.0)
    return out

def rank_biserial(diff):
    diff=np.asarray(diff,float)
    diff=diff[np.isfinite(diff)]
    diff=diff[diff!=0]
    if len(diff)==0: return 0.0
    ranks=pd.Series(np.abs(diff)).rank(method="average").to_numpy()
    pos=ranks[diff>0].sum()
    neg=ranks[diff<0].sum()
    return float((pos-neg)/(pos+neg)) if (pos+neg)>0 else 0.0

def bootstrap_mean_ci(diff,n_boot=5000,seed=2026):
    diff=np.asarray(diff,float)
    rng=np.random.default_rng(seed)
    means=np.empty(n_boot)
    for i in range(n_boot):
        means[i]=rng.choice(diff,size=len(diff),replace=True).mean()
    return float(np.percentile(means,2.5)),float(np.percentile(means,97.5))

# Average each trainable model across seeds at SERIES level before paired testing.
series_mean=(
    main_series_seed
    .groupby(["dataset","model","unique_id","horizon"],as_index=False)
    .agg(MAE=("MAE","mean"),MASE=("MASE","mean"))
)

comparators=["SeasonalNaive168","DirectLinear","DLinear","LSTM","PatchTransformer"]
stat_rows=[]

for ds in ["BWDF","SOWEKI"]:
    for h in HORIZONS:
        s=series_mean[(series_mean.dataset==ds)&(series_mean.horizon==h)]
        piv=s.pivot(index="unique_id",columns="model",values="MAE")
        local=[]

        for base in comparators:
            if "GLINT" not in piv.columns or base not in piv.columns:
                continue
            pair=piv[["GLINT",base]].dropna()
            if len(pair)<5: continue

            a=pair["GLINT"].to_numpy()
            b=pair[base].to_numpy()
            diff=a-b  # negative favors GLINT

            if np.allclose(diff,0):
                stat,p=0.0,1.0
            else:
                r=wilcoxon(a,b,alternative="two-sided",method="auto")
                stat,p=float(r.statistic),float(r.pvalue)

            lo,hi=bootstrap_mean_ci(diff)
            local.append({
                "dataset":ds,"horizon":h,"baseline":base,
                "n_series":len(pair),
                "mean_delta_MAE_full_minus_baseline":float(diff.mean()),
                "median_delta_MAE_full_minus_baseline":float(np.median(diff)),
                "bootstrap95_low":lo,
                "bootstrap95_high":hi,
                "wins_full":int(np.sum(a<b)),
                "ties":int(np.sum(np.isclose(a,b))),
                "losses_full":int(np.sum(a>b)),
                "rank_biserial":rank_biserial(diff),
                "wilcoxon_stat":stat,
                "p_value":p,
            })

        if local:
            adj=holm_adjust([r["p_value"] for r in local])
            for r,pa in zip(local,adj):
                r["p_holm"]=float(pa)
                r["significant_0.05"]=bool(pa<.05)
                stat_rows.append(r)

stats=pd.DataFrame(stat_rows)
stats.to_csv(OUT_DIR/"SERIES_LEVEL_STATISTICS_FINAL.csv",index=False)
display(stats.round(6))

In [ ]:
# ============================================================
# 10. EFFICIENCY + GATE INTERPRETATION + FIGURES
# ============================================================

eff_all=pd.concat([efficiency,ablation_efficiency],ignore_index=True)

eff_summary=(
    eff_all
    .groupby(["dataset","model"],as_index=False)
    .agg(
        parameters=("parameters","max"),
        train_seconds_mean=("train_seconds","mean"),
        train_seconds_sd=("train_seconds","std"),
        ms_per_window_mean=("ms_per_window","mean"),
        ms_per_window_sd=("ms_per_window","std"),
        windows_per_second_mean=("windows_per_second","mean"),
        peak_train_gpu_mb_mean=("peak_train_gpu_mb","mean"),
        peak_inference_gpu_mb_mean=("peak_inference_gpu_mb","mean"),
    )
)
eff_summary.to_csv(OUT_DIR/"COMPUTATIONAL_EFFICIENCY_FINAL.csv",index=False)
display(eff_summary.round(4))

if len(gate_windows):
    gate_cols=[c for c in gate_windows.columns if c.startswith("w_")]
    gate_summary=(
        gate_windows.groupby(["dataset","horizon"],as_index=False)[gate_cols].mean()
    )
    gate_summary.to_csv(OUT_DIR/"GATE_WEIGHTS_FINAL.csv",index=False)
    display(gate_summary.round(4))

# Main MASE figure
for ds in ["BWDF","SOWEKI"]:
    z=main_results[main_results.dataset==ds].copy()
    plt.figure(figsize=(9,5))
    for model in ["SeasonalNaive168","DirectLinear","DLinear","LSTM","PatchTransformer","GLINT"]:
        q=z[z.model==model].sort_values("horizon")
        if len(q):
            plt.plot(q.horizon,q.MASE,marker="o",label=model)
    plt.xlabel("Forecast horizon (hours)")
    plt.ylabel("Macro MASE")
    plt.title(f"{ds}: multi-horizon forecasting performance")
    plt.grid(alpha=.25)
    plt.legend(ncol=2)
    plt.tight_layout()
    plt.savefig(FIG_DIR/f"{ds.lower()}_main_mase.png",dpi=220)
    plt.show()

# Ablation figure
for ds in ["BWDF","SOWEKI"]:
    z=pd.concat([
        main_results[(main_results.dataset==ds)&(main_results.model=="GLINT")],
        abl_results[abl_results.dataset==ds]
    ])
    plt.figure(figsize=(10,5))
    for model in z.model.unique():
        q=z[z.model==model].sort_values("horizon")
        plt.plot(q.horizon,q.MASE,marker="o",label=model)
    plt.xlabel("Forecast horizon (hours)")
    plt.ylabel("Macro MASE")
    plt.title(f"{ds}: GLINT ablation")
    plt.grid(alpha=.25)
    plt.legend(ncol=2)
    plt.tight_layout()
    plt.savefig(FIG_DIR/f"{ds.lower()}_ablation_mase.png",dpi=220)
    plt.show()

# Gate figure
if len(gate_windows):
    for ds in ["BWDF","SOWEKI"]:
        z=gate_summary[gate_summary.dataset==ds]
        plt.figure(figsize=(9,5))
        for c in [c for c in z.columns if c.startswith("w_")]:
            plt.plot(z.horizon,z[c],marker="o",label=c.replace("w_",""))
        plt.xlabel("Forecast horizon (hours)")
        plt.ylabel("Mean expert weight")
        plt.ylim(0,1)
        plt.title(f"{ds}: learned expert weights")
        plt.grid(alpha=.25)
        plt.legend()
        plt.tight_layout()
        plt.savefig(FIG_DIR/f"{ds.lower()}_gate_weights.png",dpi=220)
        plt.show()

# ------------------------------------------------------------
# Forecast example bank — ALL horizons
# ------------------------------------------------------------
forecast_example_summary=pd.DataFrame()

if len(examples):
    summary_rows=[]

    grouped=examples.groupby(
        ["dataset","unique_id","origin","horizon"],
        sort=True
    )

    for (ds,sid,origin,h),g in grouped:
        g=g.sort_values("step").copy()
        valid=g["observed"].astype(bool)

        if valid.sum()==0:
            continue

        yt=g.loc[valid,"actual"].to_numpy()
        yp=g.loc[valid,"prediction"].to_numpy()
        ys=g.loc[valid,"seasonal_naive"].to_numpy()

        mae=float(np.mean(np.abs(yp-yt)))
        rmse=float(np.sqrt(np.mean((yp-yt)**2)))
        smape=float(200*np.mean(
            np.abs(yp-yt)/(np.abs(yt)+np.abs(yp)+1e-8)
        ))

        seasonal_mae=float(np.mean(np.abs(ys-yt)))
        improvement_vs_seasonal=(
            100*(seasonal_mae-mae)/seasonal_mae
            if seasonal_mae>1e-12 else np.nan
        )

        summary_rows.append({
            "dataset":ds,
            "unique_id":sid,
            "origin":origin,
            "horizon":int(h),
            "n_observed":int(valid.sum()),
            "MAE":mae,
            "RMSE":rmse,
            "sMAPE":smape,
            "SeasonalNaive_MAE":seasonal_mae,
            "improvement_vs_seasonal_pct":improvement_vs_seasonal,
        })

        # Save every candidate figure to disk; do not display hundreds inline.
        fig_path=FIG_DIR/"forecast_bank"
        fig_path.mkdir(parents=True,exist_ok=True)

        plt.figure(figsize=(10,4))
        plt.plot(g["step"],g["actual"],label="Actual")
        plt.plot(g["step"],g["prediction"],label="GLINT")
        plt.plot(g["step"],g["seasonal_naive"],label="Seasonal Naïve",alpha=.75)
        plt.xlabel("Forecast step (hour)")
        plt.ylabel("Demand")
        plt.title(f"{ds} | {sid} | {int(h)}-h forecast")
        plt.grid(alpha=.2)
        plt.legend()
        plt.tight_layout()

        safe_origin=str(origin).replace(":","-").replace(" ","_")
        fname=f"{ds.lower()}_{sid.lower()}_h{int(h)}_{safe_origin}.png"
        plt.savefig(fig_path/fname,dpi=220)
        plt.close()

    forecast_example_summary=pd.DataFrame(summary_rows)
    forecast_example_summary.to_csv(
        OUT_DIR/"FORECAST_EXAMPLE_SUMMARY.csv",
        index=False
    )
    examples.to_parquet(
        OUT_DIR/"FORECAST_EXAMPLE_BANK.parquet",
        index=False
    )

    # Objective representative selection:
    # for each dataset × horizon choose the stored example whose MAE is
    # closest to the median stored-example MAE. This avoids cherry-picking
    # an exceptionally easy/best-looking test window.
    representative_rows=[]

    for (ds,h),z in forecast_example_summary.groupby(["dataset","horizon"]):
        med=float(z["MAE"].median())
        zz=z.assign(distance_to_median=(z["MAE"]-med).abs()).sort_values(
            ["distance_to_median","unique_id","origin"]
        )
        representative_rows.append(zz.iloc[0])

    representative_examples=pd.DataFrame(representative_rows)
    representative_examples.to_csv(
        OUT_DIR/"REPRESENTATIVE_FORECAST_EXAMPLES.csv",
        index=False
    )

    # Manuscript-ready representative plot for every dataset × horizon.
    rep_dir=FIG_DIR/"representative_forecasts"
    rep_dir.mkdir(parents=True,exist_ok=True)

    for _,r in representative_examples.iterrows():
        g=examples[
            (examples.dataset==r.dataset)
            &(examples.unique_id==r.unique_id)
            &(examples.origin==r.origin)
            &(examples.horizon==r.horizon)
        ].sort_values("step")

        plt.figure(figsize=(10,4))
        plt.plot(g["step"],g["actual"],label="Actual")
        plt.plot(g["step"],g["prediction"],label="GLINT")
        plt.plot(g["step"],g["seasonal_naive"],label="Seasonal Naïve",alpha=.75)
        plt.xlabel("Forecast step (hour)")
        plt.ylabel("Demand")
        plt.title(
            f"{r.dataset} representative {int(r.horizon)}-h forecast | "
            f"{r.unique_id}"
        )
        plt.grid(alpha=.2)
        plt.legend()
        plt.tight_layout()
        plt.savefig(
            rep_dir/f"{r.dataset.lower()}_representative_h{int(r.horizon)}.png",
            dpi=240
        )
        plt.show()

    display(
        representative_examples[
            ["dataset","horizon","unique_id","origin","MAE","RMSE",
             "SeasonalNaive_MAE","improvement_vs_seasonal_pct"]
        ].sort_values(["dataset","horizon"]).round(4)
    )

In [ ]:
# ============================================================
# 11. MANUSCRIPT-READY FIGURES + TABLE PACK
# ============================================================

MANUSCRIPT_DIR=OUT_DIR/"manuscript_pack"
MANUSCRIPT_FIG=MANUSCRIPT_DIR/"figures"
MANUSCRIPT_TAB=MANUSCRIPT_DIR/"tables"

for p in [MANUSCRIPT_DIR,MANUSCRIPT_FIG,MANUSCRIPT_TAB]:
    p.mkdir(parents=True,exist_ok=True)

# ------------------------------------------------------------
# FIGURE 1. GLINT architecture schematic
# ------------------------------------------------------------
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

plt.figure(figsize=(12,7))
ax=plt.gca()
ax.set_xlim(0,12)
ax.set_ylim(0,8)
ax.axis("off")

def box(x,y,w,h,text):
    patch=FancyBboxPatch(
        (x,y),w,h,
        boxstyle="round,pad=0.03,rounding_size=0.08",
        linewidth=1.2,
        fill=False
    )
    ax.add_patch(patch)
    ax.text(x+w/2,y+h/2,text,ha="center",va="center",fontsize=10)
    return (x,y,w,h)

def arrow(x1,y1,x2,y2):
    ax.add_patch(FancyArrowPatch(
        (x1,y1),(x2,y2),
        arrowstyle="->",
        mutation_scale=12,
        linewidth=1.1
    ))

box(4.5,7.0,3.0,.65,"336-h historical demand")
box(.5,5.55,2.3,.7,"Weekly seasonal\nexpert (168 h)")
box(3.2,5.55,2.3,.7,"Daily seasonal\nexpert (24 h)")
box(5.9,5.55,2.3,.7,"Seasonal residual\nr(t)=y(t)-y(t-168)")
box(8.8,5.55,2.6,.7,"Known-future calendar\n(hour/day encoding)")

box(5.3,3.9,2.0,.75,"Residual DLinear\nexpert")
box(7.8,3.9,2.8,.75,"Multi-scale residual\nTransformer expert")
box(4.1,2.25,4.0,.8,"Adaptive expert gating\n(step-wise mixture weights)")
box(4.25,.65,3.7,.8,"Multi-horizon forecast\n24 / 72 / 168 / 336 h")

arrow(6.0,7.0,1.65,6.25)
arrow(6.0,7.0,4.35,6.25)
arrow(6.0,7.0,7.05,6.25)
arrow(7.05,5.55,6.3,4.65)
arrow(7.05,5.55,9.2,4.65)

arrow(1.65,5.55,5.0,3.05)
arrow(4.35,5.55,5.4,3.05)
arrow(6.3,3.9,5.7,3.05)
arrow(9.2,3.9,6.6,3.05)
arrow(10.1,5.55,7.2,3.05)
arrow(6.1,2.25,6.1,1.45)

ax.text(
    6,.15,
    "GLINT: explicit seasonal priors + linear/nonlinear residual experts + adaptive fusion",
    ha="center",va="center",fontsize=10
)

plt.tight_layout()
plt.savefig(MANUSCRIPT_FIG/"Fig1_GLINT_architecture.png",dpi=300,bbox_inches="tight")
plt.savefig(MANUSCRIPT_FIG/"Fig1_GLINT_architecture.svg",bbox_inches="tight")
plt.savefig(MANUSCRIPT_FIG/"Fig1_GLINT_architecture.pdf",bbox_inches="tight")
plt.show()

# ------------------------------------------------------------
# FIGURE 2. Dataset periodicity evidence
# median normalized autocorrelation across series, lags 1..336
# ------------------------------------------------------------
def dataset_acf_table(panel,dataset_name,max_lag=336):
    rows=[]
    for sid,g in panel.groupby("unique_id"):
        y=g.sort_values("ds")["y"].astype(float).copy()
        y=y.interpolate(limit_direction="both").ffill().bfill()
        if y.std()<1e-12:
            continue
        vals=[]
        for lag in range(1,max_lag+1):
            vals.append(float(y.autocorr(lag=lag)))
        for lag,val in enumerate(vals,start=1):
            rows.append({
                "dataset":dataset_name,
                "unique_id":sid,
                "lag":lag,
                "acf":val
            })
    return pd.DataFrame(rows)

acf_bwdf=dataset_acf_table(bwdf_panel,"BWDF")
acf_soweki=dataset_acf_table(soweki_panel,"SOWEKI")
acf_all=pd.concat([acf_bwdf,acf_soweki],ignore_index=True)
acf_summary=(
    acf_all.groupby(["dataset","lag"],as_index=False)
    .agg(median_acf=("acf","median"),q25=("acf",lambda s:s.quantile(.25)),q75=("acf",lambda s:s.quantile(.75)))
)
acf_summary.to_csv(MANUSCRIPT_TAB/"dataset_periodicity_acf.csv",index=False)

for ds in ["BWDF","SOWEKI"]:
    z=acf_summary[acf_summary.dataset==ds]
    plt.figure(figsize=(10,4))
    plt.plot(z.lag,z.median_acf,label="Median ACF")
    plt.axvline(24,linestyle="--",linewidth=1,label="24 h")
    plt.axvline(168,linestyle="--",linewidth=1,label="168 h")
    plt.xlabel("Lag (hours)")
    plt.ylabel("Autocorrelation")
    plt.title(f"{ds}: periodic structure across demand series")
    plt.grid(alpha=.2)
    plt.legend()
    plt.tight_layout()
    plt.savefig(MANUSCRIPT_FIG/f"Fig2_{ds.lower()}_periodicity.png",dpi=300)
    plt.show()

# ------------------------------------------------------------
# FIGURE 6. Accuracy-efficiency trade-off
# ------------------------------------------------------------
eff_main=eff_summary[
    eff_summary.model.isin(
        ["SeasonalNaive168","DirectLinear","DLinear","LSTM","PatchTransformer","GLINT"]
    )
].copy()

performance_overall=(
    main_results.groupby(["dataset","model"],as_index=False)
    .agg(mean_MASE=("MASE","mean"))
)

accuracy_eff=performance_overall.merge(
    eff_main[["dataset","model","ms_per_window_mean","parameters",
              "train_seconds_mean","peak_inference_gpu_mb_mean"]],
    on=["dataset","model"],how="left"
)
accuracy_eff.to_csv(MANUSCRIPT_TAB/"accuracy_efficiency.csv",index=False)

for ds in ["BWDF","SOWEKI"]:
    z=accuracy_eff[accuracy_eff.dataset==ds].dropna(subset=["ms_per_window_mean","mean_MASE"])
    plt.figure(figsize=(8,5))
    plt.scatter(z.ms_per_window_mean,z.mean_MASE,s=70)
    for _,r in z.iterrows():
        plt.annotate(
            r.model,
            (r.ms_per_window_mean,r.mean_MASE),
            xytext=(5,5),
            textcoords="offset points",
            fontsize=8
        )
    plt.xlabel("Inference latency (ms/window)")
    plt.ylabel("Mean MASE across horizons")
    plt.title(f"{ds}: accuracy-efficiency trade-off")
    plt.grid(alpha=.2)
    plt.tight_layout()
    plt.savefig(MANUSCRIPT_FIG/f"Fig6_{ds.lower()}_accuracy_efficiency.png",dpi=300)
    plt.show()

# ------------------------------------------------------------
# FIGURE 7. Per-series robustness heatmap
# Positive value = GLINT improves over strongest competing baseline.
# ------------------------------------------------------------
series_avg=(
    main_series_seed
    .groupby(["dataset","model","unique_id","horizon"],as_index=False)
    .agg(MASE=("MASE","mean"))
)

robust_rows=[]
for ds in ["BWDF","SOWEKI"]:
    for sid in sorted(series_avg[series_avg.dataset==ds].unique_id.unique()):
        for h in HORIZONS:
            z=series_avg[
                (series_avg.dataset==ds)
                &(series_avg.unique_id==sid)
                &(series_avg.horizon==h)
            ]
            f=z[z.model=="GLINT"]
            b=z[z.model!="GLINT"]
            if f.empty or b.empty:
                continue
            full=float(f.MASE.iloc[0])
            best_idx=b.MASE.idxmin()
            base=float(b.loc[best_idx,"MASE"])
            base_model=str(b.loc[best_idx,"model"])
            imp=100*(base-full)/base if base>1e-12 else np.nan
            robust_rows.append({
                "dataset":ds,
                "unique_id":sid,
                "horizon":h,
                "GLINT_MASE":full,
                "best_baseline":base_model,
                "best_baseline_MASE":base,
                "improvement_pct":imp
            })

robustness=pd.DataFrame(robust_rows)
robustness.to_csv(MANUSCRIPT_TAB/"per_series_robustness.csv",index=False)

for ds in ["BWDF","SOWEKI"]:
    z=robustness[robustness.dataset==ds]
    pivot=z.pivot(index="unique_id",columns="horizon",values="improvement_pct")
    pivot=pivot.reindex(columns=HORIZONS)

    plt.figure(figsize=(7,max(4,.38*len(pivot))))
    im=plt.imshow(pivot.values,aspect="auto")
    plt.colorbar(im,label="Improvement over strongest baseline (%)")
    plt.xticks(range(len(pivot.columns)),[str(x) for x in pivot.columns])
    plt.yticks(range(len(pivot.index)),pivot.index)
    plt.xlabel("Forecast horizon (hours)")
    plt.ylabel("Series")
    plt.title(f"{ds}: per-series GLINT robustness")
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val=pivot.iloc[i,j]
            if np.isfinite(val):
                plt.text(j,i,f"{val:.1f}",ha="center",va="center",fontsize=7)
    plt.tight_layout()
    plt.savefig(MANUSCRIPT_FIG/f"Fig7_{ds.lower()}_per_series_robustness.png",dpi=300)
    plt.show()

# ------------------------------------------------------------
# Manuscript tables 1–7
# ------------------------------------------------------------

# Table 1 — dataset characteristics.
table1=(
    data_quality.groupby("dataset",as_index=False)
    .agg(
        Series=("unique_id","nunique"),
        Start=("start","min"),
        End=("end","max"),
        Total_hours=("n_hours","sum"),
        Missing_pct_mean=("missing_pct","mean"),
        Missing_pct_max=("missing_pct","max"),
    )
)
split_summary=(
    coverage.groupby(["dataset","split"],as_index=False)
    .agg(Windows=("n_windows","sum"))
    .pivot(index="dataset",columns="split",values="Windows")
    .reset_index()
)
table1=table1.merge(split_summary,on="dataset",how="left")
table1.to_csv(MANUSCRIPT_TAB/"Table1_Dataset_Characteristics.csv",index=False)

# Table 2 — final model configuration.
model_rows=[]
for model_name in ["SeasonalNaive168","DirectLinear","DLinear","LSTM","PatchTransformer","GLINT"]:
    if model_name=="SeasonalNaive168":
        params=0
        desc="168-h weekly persistence"
    else:
        m=make_model(model_name)
        params=count_params(m)
        desc={
            "DirectLinear":"Direct linear multi-horizon head",
            "DLinear":"Trend/residual linear decomposition",
            "LSTM":"Sequence LSTM forecaster",
            "PatchTransformer":"Patch-based Transformer baseline",
            "GLINT":"Seasonal + residual linear/Transformer experts + adaptive gate",
        }[model_name]
    model_rows.append({
        "Model":model_name,
        "Description":desc,
        "Lookback_h":LOOKBACK,
        "Max_horizon_h":MAX_H,
        "Trainable_parameters":params,
        "Learning_rate":0 if model_name=="SeasonalNaive168" else FINAL_CFG["lr"],
        "Seeds":"deterministic" if model_name=="SeasonalNaive168" else ",".join(map(str,SEEDS)),
    })
table2=pd.DataFrame(model_rows)
table2.to_csv(MANUSCRIPT_TAB/"Table2_Model_Configuration.csv",index=False)

# Table 3 — main forecasting performance.
table3=main_results[
    ["dataset","model","horizon","MAE","MAE_seed_sd","RMSE","RMSE_seed_sd",
     "sMAPE","sMAPE_seed_sd","MASE","MASE_seed_sd","n_series"]
].copy()
table3.to_csv(MANUSCRIPT_TAB/"Table3_Main_Performance.csv",index=False)

# Table 4 — ablation.
table4=pd.concat([
    main_results[main_results.model=="GLINT"],
    abl_results
],ignore_index=True)[
    ["dataset","model","horizon","MAE","RMSE","sMAPE","MASE","MASE_seed_sd","n_series"]
]
table4.to_csv(MANUSCRIPT_TAB/"Table4_Ablation.csv",index=False)

# Table 5 — statistics.
table5=stats.copy()
table5.to_csv(MANUSCRIPT_TAB/"Table5_Statistical_Comparison.csv",index=False)

# Table 6 — computational efficiency.
table6=eff_summary.copy()
table6.to_csv(MANUSCRIPT_TAB/"Table6_Computational_Efficiency.csv",index=False)

# Table 7 — expert gate behavior.
if len(gate_windows):
    table7=gate_summary.copy()
else:
    table7=pd.DataFrame()
table7.to_csv(MANUSCRIPT_TAB/"Table7_Expert_Gate_Behavior.csv",index=False)

# One consolidated workbook for manuscript tables.
manuscript_xlsx=MANUSCRIPT_DIR/"IJIES_MANUSCRIPT_TABLES.xlsx"
with pd.ExcelWriter(manuscript_xlsx,engine="openpyxl") as w:
    table1.to_excel(w,sheet_name="Table1_Dataset",index=False)
    table2.to_excel(w,sheet_name="Table2_Models",index=False)
    table3.to_excel(w,sheet_name="Table3_Main",index=False)
    table4.to_excel(w,sheet_name="Table4_Ablation",index=False)
    table5.to_excel(w,sheet_name="Table5_Statistics",index=False)
    table6.to_excel(w,sheet_name="Table6_Efficiency",index=False)
    table7.to_excel(w,sheet_name="Table7_Gates",index=False)
    if len(forecast_example_summary):
        forecast_example_summary.to_excel(w,sheet_name="Forecast_Candidates",index=False)
        representative_examples.to_excel(w,sheet_name="Forecast_Selected",index=False)
    robustness.to_excel(w,sheet_name="PerSeries_Robustness",index=False)
    accuracy_eff.to_excel(w,sheet_name="Accuracy_Efficiency",index=False)

print("Manuscript pack:",MANUSCRIPT_DIR)
print("Manuscript workbook:",manuscript_xlsx)

In [ ]:
# ============================================================
# 12. MANUSCRIPT READINESS GATE + COMPLETE EXPORT
# ============================================================

# 8 dataset × horizon combinations.
gate_rows=[]
for ds in ["BWDF","SOWEKI"]:
    for h in HORIZONS:
        z=main_results[(main_results.dataset==ds)&(main_results.horizon==h)].sort_values("MASE")
        full=z[z.model=="GLINT"].iloc[0]
        best=z.iloc[0]
        best_base=z[z.model!="GLINT"].iloc[0]

        # Ablation competitors
        az=abl_results[(abl_results.dataset==ds)&(abl_results.horizon==h)]
        best_abl=az.sort_values("MASE").iloc[0]

        gate_rows.append({
            "dataset":ds,
            "horizon":h,
            "GLINT_MASE":float(full.MASE),
            "GLINT_seed_sd":float(full.MASE_seed_sd),
            "best_model_overall":best.model,
            "best_MASE_overall":float(best.MASE),
            "best_baseline":best_base.model,
            "best_baseline_MASE":float(best_base.MASE),
            "improvement_vs_best_baseline_pct":
                float(100*(best_base.MASE-full.MASE)/best_base.MASE),
            "GLINT_is_best":bool(best.model=="GLINT"),
            "GLINT_MASE_lt_1":bool(full.MASE<1),
            "best_ablation":best_abl.model,
            "best_ablation_MASE":float(best_abl.MASE),
            "full_better_than_all_ablations":bool(full.MASE<=best_abl.MASE),
        })

readiness=pd.DataFrame(gate_rows)
readiness.to_csv(OUT_DIR/"MANUSCRIPT_READINESS_GATE.csv",index=False)

wins=int(readiness.GLINT_is_best.sum())
under1=int(readiness.GLINT_MASE_lt_1.sum())
ablwins=int(readiness.full_better_than_all_ablations.sum())

bwdf_n=int(main_results[(main_results.dataset=="BWDF")&(main_results.model=="GLINT")].n_series.max())
soweki_n=int(main_results[(main_results.dataset=="SOWEKI")&(main_results.model=="GLINT")].n_series.max())

status=(
    "STRONG"
    if wins>=6 and under1==8 and bwdf_n==10 and soweki_n==11 and ablwins>=6
    else "REVIEW_NEEDED"
)

print("FINAL READINESS STATUS:",status)
print("GLINT best:",wins,"/8")
print("MASE < 1:",under1,"/8")
print("Full beats all ablations:",ablwins,"/8")
print("Final test series: BWDF",bwdf_n,"SOWEKI",soweki_n)
display(readiness.round(5))

# Workbook
xlsx=OUT_DIR/"IJIES_FINAL_VALIDATION_RESULTS.xlsx"
with pd.ExcelWriter(xlsx,engine="openpyxl") as w:
    data_quality.to_excel(w,sheet_name="Dataset_Quality",index=False)
    coverage.to_excel(w,sheet_name="Window_Coverage",index=False)
    main_results.to_excel(w,sheet_name="Main_Results",index=False)
    main_seed_macro.to_excel(w,sheet_name="Seed_Results",index=False)
    abl_results.to_excel(w,sheet_name="Ablation",index=False)
    stats.to_excel(w,sheet_name="Statistics",index=False)
    eff_summary.to_excel(w,sheet_name="Efficiency",index=False)
    readiness.to_excel(w,sheet_name="Readiness_Gate",index=False)
    if len(gate_windows):
        gate_summary.to_excel(w,sheet_name="Gate_Weights",index=False)
    robustness.to_excel(w,sheet_name="PerSeries_Robustness",index=False)
    accuracy_eff.to_excel(w,sheet_name="Accuracy_Efficiency",index=False)
    if len(forecast_example_summary):
        forecast_example_summary.to_excel(w,sheet_name="Forecast_Candidates",index=False)
        representative_examples.to_excel(w,sheet_name="Forecast_Selected",index=False)

# Auto brief
lines=[
    "# IJIES Final Validation Brief",
    "",
    f"Generated: {datetime.now(timezone.utc).isoformat()}",
    f"Status: **{status}**",
    "",
    "## Protocol",
    "- BWDF: 10 DMA series required.",
    "- SOWEKI: 11 series required.",
    "- 3 neural seeds: 42, 123, 456.",
    "- Frozen C1_small GLINT configuration; no test-set tuning.",
    "- Missing inputs handled causally; missing targets masked.",
    "",
    "## Readiness",
    f"- GLINT best in **{wins}/8** dataset×horizon combinations.",
    f"- GLINT MASE < 1 in **{under1}/8** combinations.",
    f"- Full GLINT better than every ablation in **{ablwins}/8** combinations.",
    f"- Test series included: BWDF **{bwdf_n}/10**, SOWEKI **{soweki_n}/11**.",
    "",
    "## Claim guardrails",
    "- Use 'cross-dataset consistency', not broad universal generalizability.",
    "- Claim statistical significance only where Holm-adjusted p < 0.05.",
    "- If an ablation equals or beats the full model, revise the claimed component contribution.",
    "- Report seed mean ± SD for trainable models.",
]
for _,r in readiness.iterrows():
    lines.append(
        f"- {r.dataset} H={int(r.horizon)}: GLINT MASE={r.GLINT_MASE:.4f}, "
        f"best baseline={r.best_baseline} ({r.best_baseline_MASE:.4f}), "
        f"improvement={r.improvement_vs_best_baseline_pct:.2f}%."
    )

brief=OUT_DIR/"IJIES_FINAL_VALIDATION_BRIEF.md"
brief.write_text("\n".join(lines),encoding="utf-8")

print("\nSaved:",xlsx)
print("Saved:",brief)
print("All final outputs:",OUT_DIR)
print("Manuscript-ready pack:",MANUSCRIPT_DIR)

## Files to send back after the complete final run

Please send these first:

1. `final_outputs/MANUSCRIPT_READINESS_GATE.csv`
2. `final_outputs/IJIES_FINAL_VALIDATION_RESULTS.xlsx`
3. `final_outputs/manuscript_pack/IJIES_MANUSCRIPT_TABLES.xlsx`

The notebook also generates a reusable **forecast-example bank for all four horizons
(24 / 72 / 168 / 336 h)**, objective representative examples, architecture artwork,
periodicity plots, main-performance figures, ablation plots, expert-gate figures,
accuracy–efficiency plots, and per-series robustness heatmaps.

No additional model re-run should be necessary for manuscript visualization if the final
evidence passes the readiness gate.